# 线性代数

## 基础知识

标量、向量、矩阵的基本计算，可以分成：

- **数值计算（对应元素操作）**：对应代码里的 +、-、*、/。**只管同位数字**，不管行列结构。
- **线性代数计算（结构映射操作）**：对应代码里的 @、dot、matmul。**必须满足行列规则**，用于空间变换。

### 代码速查表（PyTorch 写法）

| 运算类型 | 数学叫法 | PyTorch 代码 | 结果形状/数值特点 |
| :--- | :--- | :--- | :--- |
| **逐元素乘** | 哈达玛积（Hadamard） | a * b | 形状必须一致；结果形状不变 |
| **向量点积** | 内积（Inner Product） | torch.dot(a, b) | 必须一维且长度相同；结果是 **标量** |
| **矩阵乘法** | 叉乘（Matrix Product） | a @ b 或 torch.mm | 满足 (m,n) @ (n,p) = (m,p)；结果是**矩阵** |
| **矩阵×向量** | 线性变换 | A @ v | (m,n) @ (n,) = (m,)；结果是**向量** |
| **转置** | 行列互换 | a.T | 形状从 (m,n) 变 (n,m) |

1. 如果要做**特征融合 / 降维升维**（全连接层、注意力机制）—— 用 **@（矩阵乘法）**，它会自动帮你把行列配平。
2. 如果要做**数据筛选 / 缩放**（Dropout、Mask、归一化）—— 用 **（逐元素乘）**，它只改数值不改结构。
3. 如果是**求相似度**（两个向量的夹角）—— 用 **dot（点积）**，得到一个数字，衡量方向是否一致。

### 标量（单独一个数）
- **加减乘除**：标量直接跟任何形状的张量运算，自动**广播（复制）**到每个位置。
- **例子**：标量 2 + 向量 [1,2] = [3,4]；标量 3 * 矩阵 [[1,2],[3,4]] = [[3,6],[9,12]]。
- **线性代数意义**：就是**缩放**，把整个矩阵拉长或缩短。

### 向量（一维数组）：点积
向量之间的运算，最核心的区别在于**点积**（dot）和**逐元素乘**（*）：

- **加减法**：必须长度相同（或可广播），对应位置加减。[1,2] + [3,4] = [4,6]。
- **数乘（标量×向量）**：每个元素乘标量，属于数值计算。
- **逐元素乘（*）**：对应位置相乘，结果还是向量。[1,2] * [3,4] = [3,8]（哈达玛积）。
- **点积（dot / 内积）**：**这是向量的灵魂**。**对应位置相乘后全部求和**，结果是一个**标量（数字）**。  
  [1,2] · [3,4] = 1*3 + 2*4 = 3 + 8 = 11。  
  这个数字衡量了两个向量的“方向相似度”（夹角余弦的亲戚），在深度学习里用来算**注意力分数**和**线性层的加权和**。

### 矩阵（二维表格）：矩阵乘法
矩阵的核心计算是**线性变换**，也就是你之前问的“行乘列”：

- **加减法**：必须形状完全一样，对应位置相加减。
- **逐元素乘（*）**：必须形状一样或可广播，对应位置相乘（哈达玛积），常用于**掩码（Mask）**遮挡无效位置。
- **矩阵乘法（@ / matmul）**：**左行 × 右列 求和**。  
  结合你之前的 a (3,1) 和 b (1,2)：  
  - a @ b：左边列数(1) = 右边行数(1)，合法，结果是 **(3, 2)**。  
    算第一个数：a[0,0]*b[0,0] = 0*0=0；算第二个数：a[0,0]*b[0,1] = 0*1=0……最后结果是 3 行 2 列的矩阵。  
    这是**把 1 维空间映射到 2 维空间**。
- **转置（.T）**：把行列互换。比如 a 是 (3,1)，转置 a.T 就是 (1,3)。转置就是为了让形状**满足矩阵乘法的匹配条件**（比如 (3,1) 不能左乘 (3,3)，但转置成 (1,3) 就可以）。

### 矩阵 × 向量（最常见的混合运算）
这是神经网络的**基本单元**（全连接层 Wx + b）：

- **规则**：把向量当成“只有 1 列的矩阵”。要求矩阵的列数 = 向量的长度。
- **手算**：矩阵的**每一行**与向量做**点积**（对应乘求和），结果是一个新向量。
- **物理意义**：把低维向量“投影”到高维空间，或者高维压缩到低维。

## 标量
标量由只有一个元素的张量表示

In [1]:
import torch

x=torch.tensor(3.0)
y = torch.tensor(2.0)

x+y,x*y,x/y,x**y

(tensor(5.), tensor(6.), tensor(1.5000), tensor(9.))

## 向量
向量是标量值组成的列表，标量称为向量的元素

In [2]:
x = torch.arange(4)
x

tensor([0, 1, 2, 3])

In [3]:
x[3]     #索引访问

tensor(3)

- 向量的长度称为向量的**维度**
- 张量的**维度**表示张量具有的轴数

In [5]:
len(x)
# len()访问张量长度

4

In [6]:
x.shape
# .shape 访问向量长度（向量仅一个维度时）

torch.Size([4])

## 矩阵
矩阵A由m行和n列的实值标量组成

In [7]:
A = torch.arange(20).reshape(5,4)
A

tensor([[ 0,  1,  2,  3],
        [ 4,  5,  6,  7],
        [ 8,  9, 10, 11],
        [12, 13, 14, 15],
        [16, 17, 18, 19]])

In [8]:
#按索引或切片访问
A[1,2]

tensor(6)

In [12]:
A[:,0:2]

tensor([[ 0,  1],
        [ 4,  5],
        [ 8,  9],
        [12, 13],
        [16, 17]])

矩阵的转置是交换其行和列

In [13]:
A.T

tensor([[ 0,  4,  8, 12, 16],
        [ 1,  5,  9, 13, 17],
        [ 2,  6, 10, 14, 18],
        [ 3,  7, 11, 15, 19]])

对称矩阵与其转置矩阵相同

In [15]:
B=torch.tensor([[1,2,3],[2,0,4],[3,4,5]])
B

tensor([[1, 2, 3],
        [2, 0, 4],
        [3, 4, 5]])

In [16]:
B == B.T

tensor([[True, True, True],
        [True, True, True],
        [True, True, True]])

## 张量
张量描述具有任意数量轴的n维数组

In [17]:
x=torch.arange(24).reshape(2,3,4)
x

tensor([[[ 0,  1,  2,  3],
         [ 4,  5,  6,  7],
         [ 8,  9, 10, 11]],

        [[12, 13, 14, 15],
         [16, 17, 18, 19],
         [20, 21, 22, 23]]])

## 张量算法性质
任何**按元素计算**的操作都不会改变操作数形状

In [18]:
A=torch.arange(20,dtype=torch.float32).reshape(5,4)
B=A.clone()
#分配新内存，将A的副本分配给B
A,A+B

(tensor([[ 0.,  1.,  2.,  3.],
         [ 4.,  5.,  6.,  7.],
         [ 8.,  9., 10., 11.],
         [12., 13., 14., 15.],
         [16., 17., 18., 19.]]),
 tensor([[ 0.,  2.,  4.,  6.],
         [ 8., 10., 12., 14.],
         [16., 18., 20., 22.],
         [24., 26., 28., 30.],
         [32., 34., 36., 38.]]))

两个矩阵的按元素乘法称为**Hadamard积**（Hadamard product）（数学符号⊙）

In [19]:
A*B

tensor([[  0.,   1.,   4.,   9.],
        [ 16.,  25.,  36.,  49.],
        [ 64.,  81., 100., 121.],
        [144., 169., 196., 225.],
        [256., 289., 324., 361.]])

张量乘以或加上一个标量是将每个元素都和标量相乘或相加，矩阵形状不变

In [20]:
a=2
X=torch.arange(24).reshape(2,3,4)
a+X,(a*X).shape

(tensor([[[ 2,  3,  4,  5],
          [ 6,  7,  8,  9],
          [10, 11, 12, 13]],
 
         [[14, 15, 16, 17],
          [18, 19, 20, 21],
          [22, 23, 24, 25]]]),
 torch.Size([2, 3, 4]))

## 降维
求和可以沿轴降低张量的维度

In [23]:
x=torch.arange(4,dtype=torch.float32)
x,x.sum()

(tensor([0., 1., 2., 3.]), tensor(6.))

In [25]:
A.shape,A.sum(),A.sum().shape

(torch.Size([5, 4]), tensor(190.), torch.Size([]))

In [26]:
A_sum_axis0 = A.sum(axis=0)
A_sum_axis0,A_sum_axis0.shape

(tensor([40., 45., 50., 55.]), torch.Size([4]))

In [27]:
A_sum_axis1 = A.sum(axis=1)
A_sum_axis1,A_sum_axis1.shape

(tensor([ 6., 22., 38., 54., 70.]), torch.Size([5]))

In [28]:
A.sum(axis=[0,1])

tensor(190.)

平均值 = 总和 / 元素总数

In [29]:
A.mean(),A.sum()/A.numel()

(tensor(9.5000), tensor(9.5000))

In [31]:
A.mean(axis=0),A.sum(axis=0)/A.shape[0]

(tensor([ 8.,  9., 10., 11.]), tensor([ 8.,  9., 10., 11.]))

非降维求和，计算总和时保持轴数不变

In [33]:
A.shape

torch.Size([5, 4])

In [35]:
#保持两个轴
sum_A = A.sum(axis=1,keepdim=True)
sum_A

tensor([[ 6.],
        [22.],
        [38.],
        [54.],
        [70.]])

In [ ]:
A / sum_A
#广播机制

tensor([[0.0000, 0.1667, 0.3333, 0.5000],
        [0.1818, 0.2273, 0.2727, 0.3182],
        [0.2105, 0.2368, 0.2632, 0.2895],
        [0.2222, 0.2407, 0.2593, 0.2778],
        [0.2286, 0.2429, 0.2571, 0.2714]])

In [41]:
A,A.sum(axis=0)

(tensor([[ 0.,  1.,  2.,  3.],
         [ 4.,  5.,  6.,  7.],
         [ 8.,  9., 10., 11.],
         [12., 13., 14., 15.],
         [16., 17., 18., 19.]]),
 tensor([40., 45., 50., 55.]))

In [ ]:
# cumsum不会沿任何轴降低维度
# 在原有矩阵上按指定维度逐行相加至最后一行
A.cumsum(axis=0)

tensor([[ 0.,  1.,  2.,  3.],
        [ 4.,  6.,  8., 10.],
        [12., 15., 18., 21.],
        [24., 28., 32., 36.],
        [40., 45., 50., 55.]])

## 点积(dot)
向量点积是按**相同位置元素相乘再求和**

In [43]:
y = torch.ones(4,dtype=torch.float32)
x,y,torch.dot(x,y)

(tensor([0., 1., 2., 3.]), tensor([1., 1., 1., 1.]), tensor(6.))

In [44]:
torch.sum(x*y)

tensor(6.)

## 矩阵-向量积 (mv)
矩阵 m×n，向量 n，矩阵向量积 m

In [45]:
A.shape,x.shape,torch.mv(A,x)

(torch.Size([5, 4]), torch.Size([4]), tensor([ 14.,  38.,  62.,  86., 110.]))

## 矩阵-矩阵乘法(mm)
A矩阵是 n×k，矩阵B是 k×m，AB是n×m

In [46]:
B=torch.ones(4,3)
A.shape,B.shape,torch.mm(A,B)

(torch.Size([5, 4]),
 torch.Size([4, 3]),
 tensor([[ 6.,  6.,  6.],
         [22., 22., 22.],
         [38., 38., 38.],
         [54., 54., 54.],
         [70., 70., 70.]]))

## 范数
向量范数是将向量映射到标量的函数f
- 范数会因向量缩放而对应变化
- f(x+y)<=f(x)+f(y)
- 范数非负，f(x)>=0

**L2范数**：向量元素平方和的平方根

In [47]:
u = torch.tensor([3.0, -4.0])
torch.norm(u)

tensor(5.)

**L1范数**：绝对值函数 + 按元素求和

In [48]:
torch.abs(u).sum()

tensor(7.)

**Frobenius范数**：矩阵元素平方和的平方根（类似矩阵形向量的L2范数）

In [49]:
torch.norm(torch.ones((4,9)))

tensor(6.)